In [1]:
%load_ext autoreload
    
%autoreload 2
%matplotlib inline

In [2]:
from src.phaseA import phaseA1, phaseA2, phaseA3
from src.phaseB import phaseB
from src.generators.image_generation.RuleBasedGenerator import RuleBasedGenerator

import matplotlib.pyplot as plt

In [3]:
path_to_imgs = "data/scanned/*" #DENV_imgs/*"
scanned_path = "data/scanned/"
display= False

In [4]:
# Phase A.1 - Scanning images
Images = phaseA1(
    path_to_imgs, scanned_path,
    display=display, 
    do_white_balance=True,
    is_pre_scanned=True
)

# of Images to be analyzed: 7



In [5]:
# Phase A.2 - Grids
Grids = phaseA2(Images, display=display)

In [6]:
print(len(Grids))

7


In [1]:
# save test results
results = phaseB(Grids, display=display)

NameError: name 'phaseB' is not defined

In [9]:
# Phase A.3 - Position Graph
graphs = phaseA3(Grids, display=display)

In [10]:
# --- Image Generation --- #
RBG = RuleBasedGenerator(graphs, results)

starting_indexes = [(2,4)]

RBG.setup(starting_indexes)

Folder /home/flyingnimbus2/Desktop/chimera/AmpliVision/PhaseAB/data/generated_images/blank cleared.
Folder /home/flyingnimbus2/Desktop/chimera/AmpliVision/PhaseAB/data/generated_images/final cleared.
setup completed in 0.01 seconds


In [11]:
targets=["breast", "ovarian", "prostate"]

for img in RBG.generate(n=2, targets=targets, rotation=1, noise=0, rgb = True, save = False):
    plt.imshow(img)
    plt.show()

-------------------------------------------------- 
Rule Based Generator
 --------------------------------------------------


: 

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import datasets, layers, models

In [ ]:
from tensorflow.python.client import device_lib
device_lib.list_local_devices()

In [ ]:
n = 100

targets=["breast", "ovarian", "prostate"]
train_labels = [targets[i % len(targets)] for i in range(n * len(targets))]
test_labels = [targets[i % len(targets)] for i in range(int(n/10) * len(targets))]

In [ ]:
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(10))
model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    RBG.generate(
        n=1,
        targets=targets,
        rotation=1,
        noise=0.05,
        rgb = True,
        save = False
    ), 
    train_labels, 
    epochs=10, 
    validation_data=(
        RBG.generate(n=1, targets=targets, rotation=1, noise=0.07, rgb = True, save = False),
        test_labels
    )
)